# Sistema de Apoio ao Diagnóstico de Câncer do Colo do Útero
Notebook completo para apresentação

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import *
from sklearn.inspection import permutation_importance

In [ ]:
BASE_DIR = os.getcwd()
DATA_PATH = os.path.join(BASE_DIR, 'data', 'kag_risk_factors_cervical_cancer.csv')
print(os.listdir('data'))

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.replace('?', np.nan)
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df.head()

In [ ]:
TARGET = 'Biopsy'
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)
y.value_counts()

In [ ]:
sns.countplot(x=y)
plt.title('Distribuição da variável alvo')
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
df['Age'].dropna().hist(bins=25)
plt.title('Distribuição de idade')
plt.show()

In [ ]:
miss = df.isna().sum().sort_values(ascending=False).head(10)
miss.plot(kind='barh')
plt.title('Valores ausentes')
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

In [ ]:
preprocess = ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median')),('scaler', StandardScaler())]), X.columns)])

In [ ]:
models = {'Logistic': Pipeline([('prep', preprocess),('model', LogisticRegression(max_iter=2000))]),'RF': Pipeline([('prep', preprocess),('model', RandomForestClassifier())])}

In [ ]:
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print('\n', name)
    print(classification_report(y_test, y_pred))

In [ ]:
best_model = models['RF']
cm = confusion_matrix(y_test, best_model.predict(X_test))
sns.heatmap(cm, annot=True, fmt='d')
plt.title('Matriz de confusão')
plt.show()

In [ ]:
perm = permutation_importance(best_model, X_test, y_test)
imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
imp.head(10).plot(kind='barh')
plt.title('Importância das variáveis')
plt.show()

## Conclusão
Modelo de apoio à decisão médica com foco em recall.